In [ ]:
import os 

os.chdir("..")

In [ ]:
from CensusForge import CensusAPI
from prpop_repl import DataUtils
import bambi as bmb
import arviz_plots as azp
import arviz as az
import pandas as pd
import numpy as np
import polars as pl
azp.style.use("arviz-variat")

capi = CensusAPI()

dutils = DataUtils()

In [ ]:
df = dutils.pull_dp03().filter(pl.col("state") == 72).to_pandas()
df["log_population"] = np.log(df["total_population"])
df

In [ ]:


# Step 2: Fit log-linear model with county fixed effects
model = bmb.Model(
    "np.log(total_population) ~ year + C(county)",
    df
)
results = model.fit()

In [ ]:
az.plot_trace(results);

In [ ]:
model = bmb.Model("total_population ~ year", df)
results = model.fit()

In [ ]:
az.plot_trace(results)

In [ ]:
az.summary(results)

In [ ]:
import pymc as pm
import numpy as np

X = df["year"].values
Y = df["total_population"].values

with pm.Model() as nonlinear_model:
    
    a = pm.Normal("a", 0, 10)
    b = pm.Normal("b", 0, 10)
    c = pm.Normal("c", 0, 10)
    f = pm.Normal("f", 0, 10)
    
    mu = a / (1 + b * pm.math.exp(-c * X)) + c * f
    
    sigma = pm.HalfNormal("sigma", 1)
    
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=Y)
    
    trace = pm.sample()

In [ ]:
az.plot_trace(trace)